<a href="https://colab.research.google.com/github/AlejandraSaldana/TC2008B.103/blob/main/M1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install agentpy
!pip install seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 994.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 778.9/778.9 kB 6.8 MB/s eta 0:00:00


In [7]:
# Simulacion de un robot de limpieza
# El programa modela agentes que se mueven aleatoriamente y limpian celdas asignadas como sucias
# Autor: Alejandra Saldaña Rodríguez A00838889
# Fecha de creación: 15-08-2025

import agentpy as ap, numpy as np
from IPython.display import HTML
from matplotlib import pyplot as plt
import seaborn as sns
import random

# clase que representa un agente tipo roomba
class Roomba(ap.Agent):

  def setup(self):
    # inicializa movimientos en 0
    self.moves = 0

  # comportamiento del agente en cada paso
  def step(self):
    # obtiene la posicion actual del agente
    pos = self.model.room.positions[self]
    # si la celda actual esta sucia
    if self.model.dirty[pos]:
      # cambia el valor para "limpiarla"
      self.model.dirty[pos] = False
    # genera una direccion aleatoria para que se mueva el agente
    dx, dy = random.choice([(-1,-1), (-1,0), (-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)])
    # mueve al agente
    self.model.room.move_by(self, (dx, dy))
    # incrementa el contador de movimientos
    self.moves += 1

# clase que modela el entorno de limpieza (grid, celdas sucias, agentes)
class Clean(ap.Model):

  # configura el entorno
  def setup(self):
    # inicializa los movimientos maximos
    steps = self.p.timeMax
    # calcula el numero de celdas
    self.cells = self.p.m * self.p.n
    # crea un grid con las dimensiones ingresadas
    self.room = ap.Grid(self, (self.p.m, self.p.n), check_border=True)
    # diccionario con la posicion de la celda como llave y valor inicial de False = limpia
    self.dirty = {(x,y): False for x in range(self.p.m) for y in range(self.p.n)}
    # calcula el numero de celdas sucias en base a dirtyPer
    numDirty = int(self.p.dirtyPer * self.cells / 100)
    pos = list(self.dirty.keys())
    toBeDirty = random.sample(pos, numDirty)
    # marca True en las celdas que estaran sucias
    for pos in toBeDirty:
      self.dirty[pos] = True
    # lista de agentes
    self.agents = ap.AgentList(self, self.p.numAgents, Roomba)
    # agrega los agentes al grid en la posicion inicial
    self.room.add_agents(self.agents, positions = [(1,1)] * self.p.numAgents)

  # ciclo de la simulacion
  def step(self):
    # step de los agentes
    self.agents.step()
    # si se llego al numero maximo de movimientos o esta todo limpio se detiene
    if self.t == self.p.timeMax or all(not sucia for sucia in self.dirty.values()):
      self.stop()

  # actualiza los valores
  def update(self):
    # calcula el porcentaje de celdas limpias
    clean = sum(not sucia for sucia in self.dirty.values())
    self.perClean = (clean / self.cells) * 100
    # las guarda
    self.record('perClean', self.perClean)

# funcion para visualizar la simulación
# celdas limpias: gris claro
# celdas sucias: gris oscuro
# agentes: negro
def my_plot(model, ax):
  grid = np.zeros(model.room.shape)
  for pos, sucia in model.dirty.items():
    if sucia:
      grid[pos] = 1

  for agent, pos in model.room.positions.items():
    grid[pos] = 2

  ax.clear()
  ax.imshow(grid, cmap='Greys')
  ax.set_title(f"Tiempo: {model.t}\n"
    f"Porcentaje Limpio: {model.perClean:.0f}%\n"
    f"Movimientos Totales: {sum(agent.moves for agent in model.agents)}",
    fontsize=9)

# parametros de la simulación
parameters = {
    'm': 5,
    'n': 5,
    'numAgents': 4,
    'timeMax': 200,
    'dirtyPer': 80
}

# creación y ejecución del modelo
model = Clean(parameters)

fig, ax = plt.subplots()
animation = ap.animate(model, fig, ax, my_plot)
HTML(animation.to_jshtml())
